In [ ]:
import pystac
from pystac import StacIO

import boto3

# configure boto3 to use the right profile by default if needed
# boto3.setup_default_session(profile_name="dev2")

from sar_pipeline.utils.stac import S3StacIO
from sar_pipeline.utils.aws import find_s3_filepaths_from_suffixes

StacIO.set_default(S3StacIO)

In [ ]:
# test loading a stac item from s3
url = "s3://dea-public-data-dev/experimental/baseline/ga_s1_nrb_ew_hh_hv_1/2025/03/03/S1A_EW_GRDM_1SDH_20250303T112244_20250303T112308_058139_072E6F_DB7E/s1a__EW___A_20250303T112244_stac-item.json"

In [ ]:
item = pystac.Item.from_file(url)

In [ ]:
item

In [ ]:
# search for stac items in a bucket
stac_json_list = find_s3_filepaths_from_suffixes(
    "dea-public-data-dev", "experimental", ["stac-item.json"]
)["stac-item.json"]

In [ ]:
# sample a few items to test loading them
stac_json_list = stac_json_list[:20]

In [ ]:
# load the items and print their ids to test loading from s3

loaded_items = []


def load_item(stac_json):
    return pystac.Item.from_file(f"s3://dea-public-data-dev/{stac_json}")


for i, stac_json in enumerate(stac_json_list):
    item = load_item(stac_json)
    loaded_items.append(item)
    print(f"Loaded item {i+1}/{len(stac_json_list)}: {item.id}", end="\r")

In [ ]:
# create a collection and add the loaded items to it
collection = pystac.Collection(
    id="s1-nrb-iw-collection",
    description="A sample S1 NRB IW product collection",
    extent=pystac.Extent(
        pystac.SpatialExtent([[-180, -90, 180, 90]]),
        pystac.TemporalExtent([[None, None]]),
    ),
)

collection.add_items(loaded_items)
collection.normalize_hrefs("./stac_collection");

In [ ]:
# create a catalog and add the collection to it
catalog = pystac.Catalog(
    id="s1-nrb-iw-catalog", description="A sample catalog for S1 NRB IW products"
)
catalog.add_child(collection)

In [ ]:
# normalize and save the catalog to a local directory
catalog.normalize_and_save(
    root_href="./stac_catalog", catalog_type=pystac.CatalogType.SELF_CONTAINED
)